# 02 · Classical arm: tuned XGBoost (task T4)

XGBoost at every fraud rate (20%, 5%, 1%, 0.5%, natural 0.172%) in three variants:

- **plain**: ordinary fit, with the decision threshold chosen on validation afterwards
- **spw**: `scale_pos_weight` (frauds weighted more in the loss), with the weight tuned
- **smote**: synthetic frauds added to the training rows only

Each (variant, fraud rate) gets 50 Optuna trials on seed 0, scored on validation PR-AUC. The best settings are then refit on all 5 seeds and scored once on test.

**Use a T4 GPU runtime**: XGBoost on the 170K-row natural level is several times faster on GPU. Expect about 30–60 minutes in total. Every finished model is saved to Drive straight away, so if Colab disconnects, just **Run all** again and it continues where it stopped.

In [ ]:
# 1. Mount Drive (splits are read from it, results are written to it).
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 2. Latest code and pinned libraries.
import os, sys
REPO_URL = "https://github.com/Pakkabhabad18/tabular-fm-fraud-detection.git"
REPO_DIR = "/content/tabular-fm-fraud-detection"
if not os.path.exists(REPO_DIR):
    !git clone -q {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
!pip install -q -r {REPO_DIR}/requirements.txt
sys.path.insert(0, f"{REPO_DIR}/src")

In [ ]:
# 3. Unit tests first (now includes the XGBoost tests). Stop if anything fails.
!cd {REPO_DIR} && PYTHONPATH=src python -m pytest tests -q

In [ ]:
# 4. Load ULB and the saved splits, and check they are the exact splits on record.
#    If the fingerprint doesn't match the one committed in results/, the comparison
#    with the foundation models later would not be on the same rows.
from fraudicl.config import DRIVE_DIR
from fraudicl.data import load_ulb, features_and_target, load_splits, fingerprint

X, y = features_and_target(load_ulb())
splits = load_splits(f"{DRIVE_DIR}/splits/ulb_splits.npz")
expected = open(f"{REPO_DIR}/results/ulb_splits_fingerprint.txt").readline().strip()
assert fingerprint(splits) == expected, "splits on Drive differ from the ones on record"
print(len(splits), "splits loaded, fingerprint", expected, "matches")

In [ ]:
# 5. The sweep. Prints one line per tuning job and per fitted seed.
#    Safe to stop and rerun: finished work is skipped.
from fraudicl.experiments import run_xgb_sweep
OUT = f"{DRIVE_DIR}/results"
run_xgb_sweep(
    X, y, splits, OUT,
    variants=["plain", "spw", "smote"],
    levels=["20", "5", "1", "0.5", "natural"],
    n_trials=50,
)

In [ ]:
# 6. Mean and std over the 5 seeds for each variant and fraud rate.
from fraudicl.experiments import collect, summarise
runs = collect(OUT, "xgb")
summary = summarise(runs)
summary.round(4)

**How to read this table.**

- PR-AUC is only meaningful next to the fraud rate: a random guess scores the fraud rate itself (0.20 at 20%, 0.0017 at natural). So expect PR-AUC to fall as you go down the table even if the model is just as good. The question is how fast it falls.
- Compare the three variants within a fraud rate. If plain is as good as spw and smote, reweighting and oversampling add nothing once the threshold is tuned. That is a known result for gradient-boosted trees, and worth saying at the evaluation.
- The std column is the spread over the 5 seeds, i.e. how much the number moves just from which rows were drawn. Differences smaller than that are not real.

In [ ]:
# 7. Print what goes into the repo: paste this output back to Claude.
import json, glob
print("=== summary ===")
print(summary.round(4).to_csv(index=False))
print("=== best params ===")
for f in sorted(glob.glob(f"{OUT}/xgb_params/*.json")):
    b = json.load(open(f))
    print(json.dumps({k: b[k] for k in ["variant", "level", "val_pr_auc", "best_iteration", "tuning_seconds", "params"]}))